# 1. 공격 사진 생성 + 원본 모델 확인

원본(양자화 안 한) SmolVLM-256M으로 공격 사진을 만들고, 그 사진이 원본 모델에서 답을 길게 만드는지 확인한다.
여기서 만든 `adv/*.png`를 이후 모든 양자화 경우에 **그대로 재사용**한다.

**실행 방법**
- 코랩: 런타임을 GPU로 두고 전부 실행
- 로컬(VSCode): 커널로 `.venv`를 고르고 전부 실행
- 터미널: `jupyter execute "1_공격사진_생성_및_원본모델_확인.ipynb" --inplace --timeout=-1` (결과는 이 파일의 각 셀 아래에 저장된다)

**결과가 남는 곳**
- `imgs/` 원본 사진, `adv/` 공격 사진(PNG)
- 이미 있는 공격 사진은 다시 만들지 않는다. 다시 만들려면 공격 실행 셀의 명령 끝에 `--overwrite`를 붙인다.

In [ ]:
# 코랩에서만 패키지를 설치한다. 로컬은 README의 방법으로 만든 .venv를 쓴다.
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers accelerate
    !pip install -q --force-reinstall --no-cache-dir "pillow==11.2.1"

import torch, transformers, PIL
print("torch", torch.__version__, "| transformers", transformers.__version__, "| pillow", PIL.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

In [ ]:
import os, urllib.request
os.makedirs("imgs", exist_ok=True)
urls = {
    "cat.jpg":  "http://images.cocodataset.org/val2017/000000039769.jpg",
    "room.jpg": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "bus.jpg":  "http://images.cocodataset.org/val2017/000000000285.jpg",
}
for name, u in urls.items():
    if not os.path.exists(f"imgs/{name}"):
        urllib.request.urlretrieve(u, f"imgs/{name}")
print("원본 사진:", sorted(os.listdir("imgs")))

In [ ]:
%%writefile attack_smolvlm.py
# 실행: python attack_smolvlm.py --images ./imgs --out ./adv --iter 200 --eps 8
#   진행 상황은 터미널(노트북이면 셀 아래)에 출력되고, 공격 사진은 --out 폴더에 PNG로 저장된다.
#   이미 있는 공격 사진은 건너뛴다. 다시 만들려면 --overwrite 를 붙인다.
"""SmolVLM verbose-image attack (ported from Verbose Images, ICLR 2024)."""
import argparse
import os

import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
QUESTION = "Describe this image in one sentence."
IMG_SIZE = 512


def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def build_prompt(processor):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}]
    return processor.apply_chat_template(messages, add_generation_prompt=True)


def eos_token_ids(processor):
    ids = set()
    tok = processor.tokenizer
    tid = tok.convert_tokens_to_ids("<end_of_utterance>")
    if tid is not None and tid >= 0:
        ids.add(tid)
    if tok.eos_token_id is not None:
        ids.add(tok.eos_token_id)
    return sorted(ids)


def _to_float_array(pil_img):
    import numpy as np
    return np.asarray(pil_img, dtype="float32") / 255.0


@torch.no_grad()
def load_clean_image(path):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    return torch.from_numpy(_to_float_array(img)).permute(2, 0, 1).contiguous()


def _pil_from(x, delta):
    import numpy as np
    adv = torch.clamp(x + delta, 0, 1).detach().cpu().permute(1, 2, 0).numpy()
    return Image.fromarray((adv * 255).round().astype("uint8"))


def _save_png(adv_chw, path):
    import numpy as np
    arr = (adv_chw.detach().cpu().permute(1, 2, 0).numpy() * 255).round().astype("uint8")
    Image.fromarray(arr).save(path, format="PNG")   # 반드시 무손실 PNG (JPEG면 공격이 지워진다)


def make_pixel_values(x, delta, mean, std, n_tiles):
    adv = torch.clamp(x + delta, 0.0, 1.0)
    pv = ((adv - mean) / std).unsqueeze(0).unsqueeze(0)
    if n_tiles > 1:
        pv = pv.repeat(1, n_tiles, 1, 1, 1)
    return pv


@torch.no_grad()
def natural_len_of(model, processor, x, delta, prompt, device):
    enc = processor(text=prompt, images=[_pil_from(x, delta)], return_tensors="pt").to(device)
    gen = model.generate(**enc, do_sample=False, max_new_tokens=512)
    return gen.shape[1] - enc["input_ids"].shape[1]


def losses(model, base, pixel_values, answer_start, eos_ids, use_div):
    out = model(
        input_ids=base["input_ids"],
        attention_mask=base["attention_mask"],
        pixel_values=pixel_values,
        pixel_attention_mask=base.get("pixel_attention_mask"),
        output_hidden_states=use_div,
        use_cache=False,
    )
    logits = out.logits[0, answer_start - 1:-1, :].float()
    eos_logit = logits[:, eos_ids].max(dim=-1).values
    loss_eos = eos_logit.mean()
    logp = F.log_softmax(logits, dim=-1)
    uni = torch.full_like(logp, 1.0 / logp.shape[-1])
    loss_unc = F.kl_div(logp, uni, reduction="batchmean")
    if use_div:
        hs = out.hidden_states[-1][0, answer_start - 1:-1, :].float()
        loss_div = -torch.linalg.matrix_norm(hs.cpu(), ord="nuc").to(logits.device)
    else:
        loss_div = torch.zeros((), device=logits.device)
    return loss_eos, loss_unc, loss_div


def attack_one(model, processor, path, args, device, mean, std, eos_ids):
    x = load_clean_image(path).to(device)
    prompt = build_prompt(processor)
    delta = torch.zeros_like(x, requires_grad=True)
    eps, step = args.eps / 255.0, args.step / 255.0
    bad_words = [[e] for e in eos_ids]
    name = os.path.basename(path)

    def long_target():
        with torch.no_grad():
            enc = processor(text=prompt, images=[_pil_from(x, delta)], return_tensors="pt").to(device)
            gen = model.generate(**enc, do_sample=False, bad_words_ids=bad_words,
                                  min_new_tokens=args.target_len, max_new_tokens=args.target_len)
            new = gen[0, enc["input_ids"].shape[1]:]
            return processor.tokenizer.decode(new, skip_special_tokens=True).strip() or "This"

    target = long_target()
    print(f"[{name}] 시작 자연 답 길이: {natural_len_of(model, processor, x, delta, prompt, device)} 토큰")

    best_delta = delta.detach().clone()
    hits = 0
    for it in range(args.iter):
        if it > 0 and it % args.refresh == 0:
            target = long_target()
        base = processor(text=prompt + target, images=[_pil_from(x, delta.detach())],
                         return_tensors="pt").to(device)
        prompt_len = processor(text=prompt, images=[_pil_from(x, delta.detach())],
                               return_tensors="pt")["input_ids"].shape[1]
        n_tiles = base["pixel_values"].shape[1]
        pv = make_pixel_values(x, delta, mean, std, n_tiles)
        l_eos, l_unc, l_div = losses(model, base, pv, prompt_len, eos_ids, args.w_div > 0)
        loss = l_eos + args.w_unc * l_unc + args.w_div * l_div
        model.zero_grad(set_to_none=True)
        delta.grad = None
        loss.backward()
        if torch.isnan(loss) or torch.isnan(delta.grad).any():
            print(f"[{name}] it {it}: nan 발생 → 직전 정상 상태로 저장하고 중단")
            break
        with torch.no_grad():
            delta -= step * delta.grad.sign()
            delta.clamp_(-eps, eps)
            delta.copy_(torch.clamp(x + delta, 0, 1) - x)
            best_delta = delta.detach().clone()
        if it % args.refresh == 0:
            nl = natural_len_of(model, processor, x, delta, prompt, device)
            print(f"[{name}] it {it:4d} | eos_logit {l_eos.item():+.3f} | 자연 답 길이 {nl} 토큰")
            hits = hits + 1 if nl >= 500 else 0
            if hits >= 2:
                print(f"[{name}] 512 도달 → 조기 종료")
                break

    out_path = os.path.join(args.out, os.path.splitext(name)[0] + ".png")   # 무손실 PNG로 저장
    _save_png(torch.clamp(x + best_delta, 0, 1), out_path)
    saved = natural_len_of(model, processor, x, best_delta, prompt, device)
    print(f"[{name}] 저장본 실제 답 길이: {saved} 토큰  →  {out_path}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", default="./imgs")
    ap.add_argument("--out", default="./adv")
    ap.add_argument("--iter", type=int, default=200)
    ap.add_argument("--eps", type=float, default=8.0)
    ap.add_argument("--step", type=float, default=1.0)
    ap.add_argument("--target_len", type=int, default=128)
    ap.add_argument("--refresh", type=int, default=20)
    ap.add_argument("--w_unc", type=float, default=0.05)
    ap.add_argument("--w_div", type=float, default=0.0)
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)

    exts = (".jpg", ".jpeg", ".png", ".bmp")
    paths = [os.path.join(args.images, f) for f in sorted(os.listdir(args.images))
             if f.lower().endswith(exts)]
    todo = []   # 이미 만든 공격 사진은 다시 만들지 않는다(같은 사진을 모든 양자화 경우에 재사용)
    for p in paths:
        out_path = os.path.join(args.out, os.path.splitext(os.path.basename(p))[0] + ".png")
        if os.path.exists(out_path) and not args.overwrite:
            print(f"[{os.path.basename(p)}] 이미 있음 → 건너뜀 ({out_path})")
        else:
            todo.append(p)
    if not todo:
        print("새로 만들 사진 없음 (다시 만들려면 --overwrite)")
        return

    device = pick_device()
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.image_processor.do_image_splitting = False
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float32).to(device).eval()
    mean = torch.tensor([0.5, 0.5, 0.5], device=device).view(3, 1, 1)
    std = torch.tensor([0.5, 0.5, 0.5], device=device).view(3, 1, 1)
    eos_ids = eos_token_ids(processor)
    print("device:", device, "| eos ids:", eos_ids)

    print(f"{len(todo)}장 공격 시작")
    for p in todo:
        attack_one(model, processor, p, args, device, mean, std, eos_ids)
    print("완료 →", args.out)

if __name__ == "__main__":
    main()

In [ ]:
!{sys.executable} attack_smolvlm.py --images ./imgs --out ./adv --iter 200 --eps 8

## 원본 모델에서 확인

저장된 공격 사진(PNG)을 다시 읽어 원본 모델에 넣고, 원본 사진과 답 길이를 비교한다.
- `fp32`: 공격 사진을 만들 때 쓴 모델
- `bf16`: 양자화 실험에서 기준이 되는 모델
- 답 길이 512는 `max_new_tokens` 상한에 닿았다는 뜻이다.
- 노이즈 최대는 원본과 공격 사진의 픽셀 차이 최댓값(0~255 눈금)이며, `--eps 8`이면 8 이하여야 한다.

In [ ]:
import os, gc, numpy as np, torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

MID = "HuggingFaceTB/SmolVLM-256M-Instruct"
Q = "Describe this image in one sentence."
MAX_NEW = 512
dev = "cuda" if torch.cuda.is_available() else "cpu"
proc = AutoProcessor.from_pretrained(MID); proc.image_processor.do_image_splitting = False

def load512(path):
    return Image.open(path).convert("RGB").resize((512, 512), Image.BICUBIC)

@torch.no_grad()
def answer(model, path):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": Q}]}]
    t = proc.apply_chat_template(msgs, add_generation_prompt=True)
    enc = proc(text=t, images=[load512(path)], return_tensors="pt").to(dev)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
    new = out[0, enc["input_ids"].shape[1]:]
    return len(new), proc.tokenizer.decode(new, skip_special_tokens=True).strip()

files = sorted(os.listdir("imgs"))
stems = [os.path.splitext(f)[0] for f in files]
res = {}
for kind, dtype in [("fp32", torch.float32), ("bf16", torch.bfloat16)]:
    model = AutoModelForImageTextToText.from_pretrained(MID, dtype=dtype).to(dev).eval()
    for f, s in zip(files, stems):
        res[(s, kind)] = (answer(model, f"imgs/{f}"), answer(model, f"adv/{s}.png"))
    del model; gc.collect(); torch.cuda.empty_cache()

print("사진   | 노이즈 최대 | fp32 원본 → 공격 | bf16 원본 → 공격")
for f, s in zip(files, stems):
    a = np.asarray(load512(f"imgs/{f}"), dtype=np.int16)
    b = np.asarray(Image.open(f"adv/{s}.png").convert("RGB"), dtype=np.int16)
    (o32, _), (a32, _) = res[(s, "fp32")]
    (o16, _), (a16, _) = res[(s, "bf16")]
    print(f"{s:6s} | {int(np.abs(a - b).max()):3d}/255     | {o32:4d} → {a32:4d}      | {o16:4d} → {a16:4d}")

for kind in ["fp32", "bf16"]:
    hit = sum(res[(s, kind)][1][0] >= MAX_NEW for s in stems)
    print(f"{kind}: 공격 사진이 상한({MAX_NEW}토큰)에 닿은 사진 {hit}/{len(stems)}")

s = stems[0]
print(f"\n[{s}] 원본 사진의 답:", res[(s, "bf16")][0][1])
print(f"[{s}] 공격 사진의 답(앞 300자):", res[(s, "bf16")][1][1][:300])